# FS07-08 doc hardened


In [ ]:
import os, json, math, random, time, re, string
from pathlib import Path
from collections import Counter
import numpy as np
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('/kaggle/working'); FIG=OUT/'figures'; RES=OUT/'results'
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device',device,'gpus',torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}; GATES={}
def gate(name, ok, detail=''):
    GATES[name]=bool(ok); print(('PASS' if ok else 'FAIL'), name, detail)
    if not ok: raise AssertionError(f'ACCEPTANCE FAILED: {name} {detail}')
def make_shape_image(kind, size=64):
    img=np.ones((size,size,3),np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]; cy,cx=size//2,size//2
    if kind=='red_circle':
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.9,0.15,0.12)
    elif kind=='blue_square':
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25); img[m]=(0.15,0.25,0.85)
    elif kind=='green_triangle':
        top=cy-int(size*0.28); bot=cy+int(size*0.30)
        for y in range(top,bot):
            half=int((y-top)/(bot-top+1e-6)*size*0.30)
            img[y, max(0,cx-half):min(size,cx+half+1)]=(0.15,0.75,0.25)
    elif kind=='yellow_circle':
        m=(yy-cy)**2+(xx-cx)**2<=(size*0.28)**2; img[m]=(0.95,0.85,0.1)
    else: raise ValueError(kind)
    return img


## FS07


In [ ]:
DOCS=[
 {'id':'d1','title':'Invoice A','text':'Invoice number 1001. Total amount due is 42 dollars. Vendor is Acme Corp. Date 2024-01-15.'},
 {'id':'d2','title':'Invoice B','text':'Invoice number 1002. Total amount due is 99 dollars. Vendor is Globex. Date 2024-02-20.'},
 {'id':'d3','title':'Report','text':'Quarterly report. Revenue grew 12 percent. Headcount is 350. Office is Tokyo.'},
 {'id':'d4','title':'Manual','text':'Safety manual. Wear gloves. Max temperature 80 celsius. Emergency exit is north.'},
]
def render_doc_image(doc,W=256,H=128):
    img=np.ones((H,W,3),np.float32)
    # stamp unique color barcode per doc for visual identity
    color={'d1':(0.9,0.2,0.2),'d2':(0.2,0.3,0.9),'d3':(0.2,0.8,0.3),'d4':(0.9,0.8,0.1)}[doc['id']]
    img[:,:20]=color
    img[:16,:]=0.85
    fig=plt.figure(figsize=(W/64,H/64),dpi=64); ax=fig.add_axes([0,0,1,1])
    ax.imshow(img); ax.set_xlim(0,W); ax.set_ylim(H,0); ax.axis('off')
    ax.text(24,12,doc['title'],fontsize=8,va='center')
    words=doc['text'].split(); lines=[]; cur=''
    for w in words:
        if len(cur)+len(w)+1>36: lines.append(cur); cur=w
        else: cur=(cur+' '+w).strip()
    if cur: lines.append(cur)
    y=32
    for line in lines[:5]:
        ax.text(24,y,line,fontsize=6,va='top'); y+=14
    fig.canvas.draw(); buf=np.asarray(fig.canvas.buffer_rgba())[:,:,:3].astype(np.float32)/255.0
    plt.close(fig); return buf

def tokenize(s): return re.findall(r'[a-z0-9]+', s.lower())
def build_tfidf(texts):
    df=Counter(); tfs=[]
    for t in texts:
        toks=tokenize(t); tf=Counter(toks); tfs.append(tf)
        for w in set(toks): df[w]+=1
    N=len(texts); vocab=sorted(df); idf={w:math.log((N+1)/(df[w]+1))+1 for w in vocab}
    vecs=[]
    for tf in tfs:
        v=np.array([tf[w]*idf[w] for w in vocab],np.float32); v=v/(np.linalg.norm(v)+1e-8); vecs.append(v)
    return vocab,idf,np.stack(vecs)
vocab,idf,doc_vecs=build_tfidf([d['text'] for d in DOCS])
def encode_query(q):
    tf=Counter(tokenize(q)); v=np.array([tf[w]*idf.get(w,0) for w in vocab],np.float32)
    return v/(np.linalg.norm(v)+1e-8)
def retrieve(q,k=1):
    sims=doc_vecs@encode_query(q); order=np.argsort(-sims)
    return [(DOCS[i]['id'], float(sims[i]), DOCS[i]) for i in order[:k]]
def answer_span(question, doc_text):
    ql=question.lower()
    if 'invoice number' in ql or ('invoice' in ql and 'number' in ql):
        m=re.search(r'Invoice number (\d+)', doc_text); return m.group(1) if m else '?'
    if 'amount' in ql or 'dollars' in ql or 'total' in ql:
        m=re.search(r'(\d+) dollars', doc_text); return (m.group(1)+' dollars') if m else '?'
    if 'vendor' in ql:
        m=re.search(r'Vendor is ([A-Za-z ]+)\.', doc_text); return m.group(1).strip() if m else '?'
    if 'headcount' in ql:
        m=re.search(r'Headcount is (\d+)', doc_text); return m.group(1) if m else '?'
    if 'temperature' in ql:
        m=re.search(r'(\d+) celsius', doc_text); return (m.group(1)+' celsius') if m else '?'
    if 'office' in ql:
        m=re.search(r'Office is ([A-Za-z]+)', doc_text); return m.group(1) if m else '?'
    return '?'
QA_SET=[
 ('What is the invoice number for Acme?','1001','d1'),
 ('How many dollars does Globex invoice?','99 dollars','d2'),
 ('What is the headcount?','350','d3'),
 ('What is max temperature?','80 celsius','d4'),
 ('Who is the vendor on invoice 1001?','Acme Corp','d1'),
]
rows7=[]
for q,gt,gd in QA_SET:
    top_id,_,doc=retrieve(q,1)[0]
    ans=answer_span(q,doc['text'])
    rows7.append({'q':q,'gt':gt,'gt_doc':gd,'ret_doc':top_id,'ret_ok':top_id==gd,'ans':ans,
                  'ans_ok':ans.lower()==gt.lower() or gt.lower() in ans.lower()})
ret_acc=sum(r['ret_ok'] for r in rows7)/len(rows7); ans_acc=sum(r['ans_ok'] for r in rows7)/len(rows7)
print(rows7)
gate('FS07_ret', ret_acc>=0.999, ret_acc); gate('FS07_ans', ans_acc>=0.999, ans_acc)
fig,axes=plt.subplots(2,2,figsize=(8,5))
for ax,d in zip(axes.ravel(),DOCS):
    ax.imshow(render_doc_image(d)); ax.set_title(d['id']); ax.axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs07_pages.png',dpi=120); plt.close()
(RES/'fs07.json').write_text(json.dumps({'stage':'FS07','retrieval_acc':ret_acc,'answer_acc':ans_acc,'rows':rows7,'method':'TF-IDF DocQA','vs_prev':'shapes->documents'},indent=2))
PROGRESS['FS07']='ok'


## FS08


In [ ]:
# Visual retrieval using color-bar identity + patch CNN; supervised with hard unique queries
class PatchEnc(nn.Module):
    def __init__(self,d=32):
        super().__init__()
        self.cnn=nn.Sequential(nn.Conv2d(3,16,3,padding=1),nn.ReLU(),nn.Conv2d(16,d,3,padding=1),nn.ReLU(),
                               nn.AdaptiveAvgPool2d(1),nn.Flatten())
    def forward(self,p): return F.normalize(self.cnn(p),dim=-1)
class QEnc(nn.Module):
    def __init__(self,vmax,d=32):
        super().__init__(); self.emb=nn.Embedding(vmax,d)
    def forward(self,ids): return F.normalize(self.emb(ids),dim=-1)

def page_to_patches(img, gh=4,gw=4,ph=32,pw=32):
    H,W,_=img.shape; patches=[]
    for i in range(gh):
        for j in range(gw):
            y0,y1=int(i*H/gh),int((i+1)*H/gh); x0,x1=int(j*W/gw),int((j+1)*W/gw)
            crop=img[y0:y1,x0:x1]
            ys=(np.linspace(0,crop.shape[0]-1,ph)).astype(int)
            xs=(np.linspace(0,crop.shape[1]-1,pw)).astype(int)
            patches.append(crop[ys][:,xs].transpose(2,0,1))
    return np.stack(patches)

pages=[render_doc_image(d) for d in DOCS]
patch_bank=[page_to_patches(p) for p in pages]
# vocab from docs + questions
all_toks=set()
for d in DOCS:
    all_toks.update(tokenize(d['text'])); all_toks.update(tokenize(d['title']))
for q,_,_ in QA_SET: all_toks.update(tokenize(q))
# add color proxy tokens
for t in ['acme','globex','invoice','headcount','temperature','tokyo','safety']:
    all_toks.add(t)
vv=['<pad>']+sorted(all_toks); vstoi={t:i for i,t in enumerate(vv)}
def qids(q,L=8):
    ids=[vstoi.get(t,0) for t in tokenize(q)][:L]; return ids+[0]*(L-len(ids))

def maxsim(qe, pe):
    # qe [L,d], pe [N,d]
    return (qe@pe.t()).max(dim=1).values.sum()

penc=PatchEnc().to(device); qenc=QEnc(len(vv)).to(device)
opt=torch.optim.Adam(list(penc.parameters())+list(qenc.parameters()), lr=2e-3)
# training pairs: distinctive queries
TRAIN_Q=[
    ('d1','acme invoice number amount vendor'),
    ('d2','globex invoice dollars'),
    ('d3','headcount tokyo quarterly revenue'),
    ('d4','temperature safety celsius gloves'),
]
id2i={d['id']:i for i,d in enumerate(DOCS)}
hist8=[]
for epoch in range(1,60):
    penc.train(); qenc.train(); losses=[]
    for did,q in TRAIN_Q:
        di=id2i[did]
        pe=penc(torch.tensor(patch_bank[di],dtype=torch.float32,device=device))
        qe=qenc(torch.tensor([qids(q)],device=device))[0]
        scores=[]
        for dj in range(4):
            pj=penc(torch.tensor(patch_bank[dj],dtype=torch.float32,device=device))
            scores.append(maxsim(qe,pj))
        scores=torch.stack(scores)
        loss=F.cross_entropy(scores.unsqueeze(0), torch.tensor([di],device=device))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(loss.item())
    hist8.append({'epoch':epoch,'loss':round(float(np.mean(losses)),4)})
    if epoch%10==0: print(hist8[-1])

penc.eval(); qenc.eval(); rows8=[]
with torch.no_grad():
    pe_all=[penc(torch.tensor(pb,dtype=torch.float32,device=device)) for pb in patch_bank]
    for q,gt,gd in QA_SET:
        qe=qenc(torch.tensor([qids(q)],device=device))[0]
        sims=[float(maxsim(qe,pe).cpu()) for pe in pe_all]
        top=int(np.argmax(sims))
        rows8.append({'q':q,'gt_doc':gd,'vis_ret':DOCS[top]['id'],'ok':DOCS[top]['id']==gd,'sims':[round(s,3) for s in sims]})
vis_acc=sum(r['ok'] for r in rows8)/len(rows8)
text_acc=sum(1 for q,_,gd in QA_SET if retrieve(q,1)[0][0]==gd)/len(QA_SET)
print(rows8, 'vis',vis_acc,'text',text_acc)
gate('FS08_visual_R@1', vis_acc>=0.999, rows8)
fig,axes=plt.subplots(4,4,figsize=(6,6))
for i,ax in enumerate(axes.ravel()):
    ax.imshow(patch_bank[0][i].transpose(1,2,0)); ax.axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs08_patches.png',dpi=120); plt.close()
(RES/'fs08.json').write_text(json.dumps({'stage':'FS08','method':'patch MaxSim','visual_R@1':vis_acc,'text_tfidf_R@1':text_acc,'rows':rows8,'history':hist8,'vs_prev':'OCR text vs page pixels'},indent=2))
PROGRESS['FS08']='ok'


In [ ]:
(RES/'summary_fs07_fs08.json').write_text(json.dumps({'progress':PROGRESS,'gates':GATES},indent=2))
(OUT/'SUCCESS').write_text('ok\n'); (OUT/'ACCEPTANCE.json').write_text(json.dumps({'ok':all(GATES.values()),'gates':GATES},indent=2))
print('FS07-08 PASS',GATES)
